# 03. 로컬 LLM 전환 (Qwen + Ollama)

API 기반 LLM에서 로컬 LLM으로 전환하여 비용을 절감합니다.

## 로컬 LLM 옵션
| 방법 | 장점 | 단점 |
|------|------|------|
| **Ollama** | 설치 간편, LangChain 연동 쉬움 | 모델 선택 제한적 |
| **llama.cpp** | 최적화 우수 | 설정 복잡 |
| **Transformers** | 유연함, 튜닝 연계 용이 | VRAM 더 필요 |

## 권장 모델 (8GB VRAM 기준)
- `Qwen2.5-3B-Instruct` (4bit 양자화)
- `Qwen2.5-1.5B-Instruct` (더 가벼움)

---
## 1. Ollama 설치 및 설정

### Step 1: Ollama 설치
```bash
# Windows
# https://ollama.ai/download 에서 설치 파일 다운로드

# Mac
brew install ollama

# Linux
curl -fsSL https://ollama.ai/install.sh | sh
```

### Step 2: Qwen 모델 다운로드
```bash
# Qwen2.5 3B 모델 (권장, ~2GB)
ollama pull qwen2.5:3b

# 더 가벼운 버전 (VRAM 부족 시)
ollama pull qwen2.5:1.5b

# 한국어 특화 모델 (대안)
ollama pull llama3.2:3b
```

### Step 3: Ollama 서버 실행
```bash
ollama serve
# 기본 포트: http://localhost:11434
```

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath(".."))

from dotenv import load_dotenv
load_dotenv()

# Ollama 서버 확인
import requests
try:
    response = requests.get("http://localhost:11434/api/tags")
    if response.status_code == 200:
        models = response.json().get("models", [])
        print("✓ Ollama 서버 연결 성공")
        print(f"  설치된 모델: {[m['name'] for m in models]}")
        OLLAMA_AVAILABLE = True
    else:
        print("✗ Ollama 서버 응답 오류")
        OLLAMA_AVAILABLE = False
except requests.exceptions.ConnectionError:
    print("✗ Ollama 서버가 실행되지 않았습니다.")
    print("  터미널에서 'ollama serve' 명령으로 서버를 시작하세요.")
    OLLAMA_AVAILABLE = False

---
## 2. Ollama + LangChain 연동

In [ ]:
if OLLAMA_AVAILABLE:
    from langchain_community.chat_models import ChatOllama

    # Ollama LLM 초기화
    ollama_llm = ChatOllama(
        model="qwen2.5:3b",  # 설치한 모델명
        temperature=0,
        # 추가 옵션
        # num_ctx=4096,      # 컨텍스트 윈도우 크기
        # num_gpu=1,         # GPU 사용
    )

    print("Ollama LLM 초기화 완료")

In [ ]:
# 간단한 테스트
if OLLAMA_AVAILABLE:
    response = ollama_llm.invoke("Python에서 리스트를 정렬하는 방법을 간단히 설명해주세요.")
    print(response.content)

---
## 3. 로컬 LLM으로 RAG 전환

In [ ]:
from src.loaders.notion_loader import create_sample_documents
from src.embeddings.embedding_manager import EmbeddingManager
from src.vectorstore.chroma_store import ChromaVectorStore
from src.chains.rag_chain import RAGChain

# 기존 벡터 스토어 로드 (이미 생성된 경우)
embedding_manager = EmbeddingManager(provider="openai")
embeddings = embedding_manager.embeddings

vector_store = ChromaVectorStore(
    embeddings=embeddings,
    persist_directory="../data/chroma_sample",
    collection_name="sample_docs"
)

# 문서가 없으면 샘플 문서 추가
stats = vector_store.get_collection_stats()
if stats["count"] == 0:
    sample_docs = create_sample_documents()
    vector_store.from_documents(sample_docs)
    print("샘플 문서 추가 완료")
else:
    print(f"기존 벡터 스토어 로드: {stats['count']}개 문서")

In [ ]:
if OLLAMA_AVAILABLE:
    # 로컬 LLM으로 RAG 체인 구성
    retriever = vector_store.as_retriever(search_kwargs={"k": 3})

    local_rag = RAGChain(
        llm=ollama_llm,
        retriever=retriever
    )

    print("로컬 RAG 체인 준비 완료")

In [ ]:
# 로컬 RAG 테스트
if OLLAMA_AVAILABLE:
    import time

    question = "Spring Boot에서 JPA 설정은 어떻게 하나요?"

    print(f"질문: {question}\n")

    start_time = time.time()
    answer = local_rag.invoke(question)
    elapsed = time.time() - start_time

    print(f"답변: {answer}")
    print(f"\n응답 시간: {elapsed:.2f}초")

---
## 4. API vs 로컬 LLM 비교

In [ ]:
from src.llm.model_adapter import LLMAdapter
import time

# API LLM (OpenAI)
api_llm = LLMAdapter(provider="openai").llm
api_rag = RAGChain(llm=api_llm, retriever=retriever)

test_questions = [
    "REST API URL 설계 원칙을 설명해주세요.",
    "Spring Security에서 특정 URL을 인증 없이 접근하게 하려면?"
]

In [ ]:
# 성능 비교
results = []

for q in test_questions:
    print(f"\n{'='*60}")
    print(f"질문: {q}")
    print(f"{'='*60}")

    # API LLM
    start = time.time()
    api_answer = api_rag.invoke(q)
    api_time = time.time() - start

    print(f"\n[OpenAI API] ({api_time:.2f}초)")
    print(api_answer[:300] + "..." if len(api_answer) > 300 else api_answer)

    # 로컬 LLM
    if OLLAMA_AVAILABLE:
        start = time.time()
        local_answer = local_rag.invoke(q)
        local_time = time.time() - start

        print(f"\n[Ollama 로컬] ({local_time:.2f}초)")
        print(local_answer[:300] + "..." if len(local_answer) > 300 else local_answer)

        results.append({
            "question": q[:30] + "...",
            "api_time": api_time,
            "local_time": local_time
        })

In [ ]:
# 결과 요약
if results:
    print("\n" + "="*60)
    print("성능 비교 요약")
    print("="*60)
    print(f"{'질문':<35} {'API':>10} {'로컬':>10}")
    print("-"*60)
    for r in results:
        print(f"{r['question']:<35} {r['api_time']:>10.2f}s {r['local_time']:>10.2f}s")

---
## 5. LLM 어댑터 활용

`LLMAdapter`를 사용하면 API/로컬 LLM을 쉽게 전환할 수 있습니다.

In [ ]:
from src.llm.model_adapter import LLMAdapter

# OpenAI API 사용
api_adapter = LLMAdapter(provider="openai", model_name="gpt-4o-mini")
print(f"API LLM: {api_adapter.provider}")

# Ollama 로컬 사용
if OLLAMA_AVAILABLE:
    local_adapter = LLMAdapter(provider="ollama", model_name="qwen2.5:3b")
    print(f"로컬 LLM: {local_adapter.provider}")

# Claude API 사용 (선택사항)
# claude_adapter = LLMAdapter(provider="anthropic", model_name="claude-3-haiku-20240307")

In [ ]:
# 어댑터로 RAG 전환
def create_rag_with_provider(provider: str):
    """LLM 제공자를 지정하여 RAG 체인 생성"""
    adapter = LLMAdapter(provider=provider)
    return RAGChain(llm=adapter.llm, retriever=retriever)

# 사용 예시
if OLLAMA_AVAILABLE:
    # 로컬 RAG 생성
    local_rag = create_rag_with_provider("ollama")
    answer = local_rag.invoke("HTTP 상태 코드 201은 언제 사용하나요?")
    print(answer)

---
## 6. GPU 메모리 모니터링

In [ ]:
# NVIDIA GPU 사용량 확인 (CUDA 사용 시)
try:
    import subprocess
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=memory.used,memory.total,utilization.gpu",
         "--format=csv,noheader,nounits"],
        capture_output=True,
        text=True
    )
    if result.returncode == 0:
        used, total, util = result.stdout.strip().split(", ")
        print(f"GPU 메모리: {used}MB / {total}MB ({float(used)/float(total)*100:.1f}%)")
        print(f"GPU 사용률: {util}%")
    else:
        print("nvidia-smi 실행 실패")
except FileNotFoundError:
    print("NVIDIA GPU가 없거나 nvidia-smi를 찾을 수 없습니다.")
except Exception as e:
    print(f"GPU 정보 조회 실패: {e}")

---
## 7. Transformers 직접 사용 (선택사항)

Ollama 대신 HuggingFace Transformers를 직접 사용할 수도 있습니다.
LoRA 튜닝을 계획한다면 이 방식을 권장합니다.

### 필요 패키지 설치
```bash
pip install transformers torch accelerate bitsandbytes
```

In [ ]:
# Transformers 직접 사용 (주석 해제하여 실행)
# 주의: 첫 실행 시 모델 다운로드에 시간이 걸립니다 (~5GB)

# from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
# import torch

# # 4bit 양자화 설정 (VRAM 절약)
# quantization_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.float16,
#     bnb_4bit_quant_type="nf4"
# )

# model_name = "Qwen/Qwen2.5-3B-Instruct"

# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     quantization_config=quantization_config,
#     device_map="auto"
# )

# print("모델 로드 완료")

In [ ]:
# Transformers 모델로 생성
# def generate_response(prompt, max_new_tokens=256):
#     inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
#     outputs = model.generate(
#         **inputs,
#         max_new_tokens=max_new_tokens,
#         do_sample=False
#     )
#     return tokenizer.decode(outputs[0], skip_special_tokens=True)

# # 테스트
# response = generate_response("Python에서 리스트를 정렬하는 방법은?")
# print(response)

---
## 정리

### 로컬 LLM 사용 시 고려사항

| 항목 | API (OpenAI/Claude) | 로컬 (Ollama/Qwen) |
|------|---------------------|--------------------|
| 비용 | 사용량에 따라 과금 | 무료 (전기/GPU 비용) |
| 응답 속도 | 네트워크 지연 포함 | GPU 성능에 따름 |
| 품질 | GPT-4급 최고 성능 | 작은 모델 한계 있음 |
| 설정 | API 키만 필요 | 모델 다운로드 및 서버 실행 |
| 프라이버시 | 데이터 전송 필요 | 로컬에서 처리 |

### 권장 사용 시나리오
- **개발/테스트**: 로컬 LLM (비용 절감)
- **프로덕션**: API LLM (품질 보장)
- **민감 데이터**: 로컬 LLM (프라이버시)

다음 노트북에서는 **LoRA 튜닝**을 통해 로컬 모델의 도메인 특화 성능을 향상시킵니다.